In [ ]:
# general
import numpy as np
import pandas as pd
import json
import plotly.graph_objects as go
import plotly.express as px
from time import perf_counter

In [ ]:
# experiments
from swarm.precedence import PrecedenceManager

from psplib.launcher import load_psplib_data, Launcher
from psplib.launcher import get_dict_of_optimized_data_structures

## Prepare data

In [ ]:
file_name = "psplib/j120_json/j12016_6.json"
# file_name = "psplib/j120_json/j1206_1.json"

# loading data for the test
wg, contractors, work_estimator, best_known_makespan = load_psplib_data(file_name)
ods = get_dict_of_optimized_data_structures(wg, contractors)

n_tasks = len(wg.nodes)
# to get fitness metrics
launcher = Launcher(wg, contractors, work_estimator, ods)
# to convert to valid order
precedence_manager = PrecedenceManager.from_workgraph(wg)

In [ ]:
# for now, define fitness here
def fitness_function(priorities_array):
    activity_list = precedence_manager(priorities_array)
    makespan = launcher(activity_list)
    return makespan

## Simulated Annealing

In [ ]:
# [1] New meta-heuristics for the resource-constrained project scheduling problem (2011)

In [ ]:
class Solution:

    def __init__(self, activity_list):
        self.activity_list = activity_list.copy()
        self.fitness = fitness_function(activity_list)

    def move(self, n_changes=1):
        new_activity_list = self.activity_list.copy()
        places_to_change = np.random.choice(new_activity_list.size, n_changes, replace=False)
        new_activity_list[places_to_change] = np.random.random(n_changes)
        return Solution(new_activity_list)

class Temperature:

    def __init__(self, temperature, cooling_ratio):
        self.temperature = temperature
        self.cooling_ratio = cooling_ratio

    def cool(self):
        self.temperature = self.temperature * self.cooling_ratio

    def accept_new(self, fitness_old, fitness_new):
        if fitness_new <= fitness_old:
            return True

        delta = fitness_new - fitness_old
        probability = np.exp(-delta / self.temperature)
        return np.random.random() < probability

In [ ]:
full_history = []

for _ in range(10):

    solution = Solution(np.random.random(122))
    temperature = Temperature(temperature=2, cooling_ratio=0.99)
    history = []
    
    for i in range(100):
        n_changes = np.random.randint(1, 5)
        new_solution = solution.move(n_changes=n_changes)
        
        if temperature.accept_new(solution.fitness, new_solution.fitness):
            solution = new_solution
    
        temperature.cool()
        history.append(solution.fitness)
    
    full_history.append(np.array(history))

full_history = np.array(full_history)

In [ ]:
fig = px.line(full_history.T, template='plotly_white')
fig.update_traces(line_color='navy')
fig.update_layout(showlegend=False)
fig.show()